In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset
import random

dataset = load_dataset("b-mc2/sql-create-context")
train_dataset = dataset["train"]
data = random.sample(list(train_dataset), 100)

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "defog/llama-3-sqlcoder-8b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16  # important for memory
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [5]:
def build_prompt(question, context):
    return f"""You are a SQL generator.

Output ONLY a valid SQL query.

STRICT RULES:
- Do NOT output anything except SQL
- Do NOT explain
- Do NOT repeat the question
- Do NOT use DISTINCT unless explicitly required
- Do NOT use table aliases unless necessary
- Use the simplest possible SQL

Schema:
{context}

Question:
{question}

SQL:
"""

In [6]:
import re

def clean_sql(output_text):
    # Extract SQL part
    if "SQL:" in output_text:
        sql = output_text.split("SQL:")[-1]
    elif "SQL Query:" in output_text:
        sql = output_text.split("SQL Query:")[-1]
    else:
        sql = output_text

    sql = sql.strip()

    # 🚨 Remove everything after first semicolon
    if ";" in sql:
        sql = sql.split(";")[0] + ";"
    else:
        # fallback: stop at first newline
        sql = sql.split("\n")[0]

    # 🚨 Remove non-SQL garbage (assistant text etc.)
    sql = re.sub(r"(assistant:.*)", "", sql, flags=re.IGNORECASE)

    # 🚨 Remove trailing junk after SQL keywords
    sql = re.sub(r"(;).*", r"\1", sql)

    return sql.strip()

In [7]:
predictions = []

for i, sample in enumerate(data):
    question = sample["question"]
    context = sample["context"]
    ground_truth = sample["answer"]

    prompt = build_prompt(question, context)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,  # reduce length
            temperature=0.0,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predicted_sql = clean_sql(output_text)

    predictions.append({
        "question": question,
        "predicted_sql": predicted_sql,
        "ground_truth": ground_truth
    })

    if i % 50 == 0:
        print(f"Processed {i} samples")

Processed 0 samples
Processed 50 samples


In [8]:
import json

with open("predictions.json", "w") as f:
    json.dump(predictions, f, indent=4)

In [9]:
import re

def normalize_sql(sql):
    sql = sql.lower()
    
    # remove extra spaces
    sql = re.sub(r"\s+", " ", sql)
    
    # remove spaces around brackets and commas
    sql = re.sub(r"\s*,\s*", ",", sql)
    sql = re.sub(r"\(\s*", "(", sql)
    sql = re.sub(r"\s*\)", ")", sql)
    
    return sql.strip()

In [10]:
def exact_match(pred, gt):
    return normalize_sql(pred) == normalize_sql(gt)

In [11]:
def tokenize_sql(sql):
    sql = normalize_sql(sql)
    
    # split on spaces + punctuation
    tokens = re.split(r"[ ,()]", sql)
    tokens = [t for t in tokens if t]
    
    return set(tokens)

In [12]:
def proximal_score(pred, gt):
    p_tokens = tokenize_sql(pred)
    g_tokens = tokenize_sql(gt)
    
    if len(g_tokens) == 0:
        return 0.0
    
    intersection = len(p_tokens & g_tokens)
    union = len(p_tokens | g_tokens)
    
    return intersection / union

In [13]:
exact_count = 0
prox_count = 0

prox_scores = []

for item in predictions:
    pred = item["predicted_sql"]
    gt = item["ground_truth"]
    
    # Exact Match
    if exact_match(pred, gt):
        exact_count += 1
    
    # Proximal Match
    score = proximal_score(pred, gt)
    prox_scores.append(score)
    
    if score >= 0.7:
        prox_count += 1

n = len(predictions)

print(f"Total Samples: {n}")
print(f"Exact Match Accuracy: {exact_count / n:.4f}")
print(f"Proximal Match Accuracy (>=0.7): {prox_count / n:.4f}")
print(f"Average Proximal Score: {sum(prox_scores)/n:.4f}")

Total Samples: 100
Exact Match Accuracy: 0.0000
Proximal Match Accuracy (>=0.7): 0.0000
Average Proximal Score: 0.3598
